# Notebook 04: Vulnerability Score Construction

This notebook converts the final 2011–2025 county-year feature panel into a
multidimensional climate-housing vulnerability dataset. It constructs eight
sub-scores, combines them with equal weight, and assigns Low, Medium, and High
vulnerability classes for the later analytical notebooks.


## 1. Setup and Input Validation


In [2]:
from pathlib import Path
import sys


def locate_project_root(start_path=None):
    start = Path(start_path or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root containing notebooks/ and data/."
    )


PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from src.config import STUDY_END_YEAR, STUDY_START_YEAR
from src.visualization import save_table


pd.set_option("display.max_columns", 130)
pd.set_option("display.float_format", "{:,.3f}".format)

INPUT_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "florida_county_year_features_2011_2025.csv"
)
OUTPUT_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "florida_county_year_vulnerability_2011_2025.csv"
)
TABLES_DIR = PROJECT_ROOT / "results" / "tables" / "vulnerability_scoring"

if not INPUT_DATA_PATH.exists():
    raise FileNotFoundError(
        "Run cleaned Notebook 02 first. Missing input: "
        f"{INPUT_DATA_PATH}"
    )

county_year_features = pd.read_csv(
    INPUT_DATA_PATH,
    dtype={"STCOFIPS": str},
)
county_year_features["STCOFIPS"] = (
    county_year_features["STCOFIPS"].str.zfill(5)
)

duplicate_county_years = int(
    county_year_features.duplicated(["STCOFIPS", "Year"]).sum()
)
input_validation = pd.DataFrame(
    {
        "Metric": [
            "Rows",
            "Columns",
            "Counties",
            "Start year",
            "End year",
            "Duplicate county-years",
            "Missing values",
        ],
        "Value": [
            len(county_year_features),
            county_year_features.shape[1],
            county_year_features["STCOFIPS"].nunique(),
            county_year_features["Year"].min(),
            county_year_features["Year"].max(),
            duplicate_county_years,
            int(county_year_features.isna().sum().sum()),
        ],
    }
)

if county_year_features.shape != (1000, 75):
    raise ValueError("Expected the 1,000-row × 75-column feature dataset.")
if county_year_features["STCOFIPS"].nunique() != 67:
    raise ValueError("Expected 67 Florida counties.")
if not county_year_features["Year"].between(
    STUDY_START_YEAR, STUDY_END_YEAR
).all():
    raise ValueError("Observations fall outside the 2011–2025 study period.")
if duplicate_county_years:
    raise ValueError("Duplicate county-year observations found.")

display(input_validation)
display(
    county_year_features.isna().sum()
    .loc[lambda values: values.gt(0)]
    .rename("missing_values")
    .to_frame()
)


,Metric,Value
0,Rows,1000
1,Columns,75
2,Counties,67
3,Start year,2011
4,End year,2025
5,Duplicate county-years,0
6,Missing values,82


,missing_values
prev_year_housing_price,3
annual_price_growth_dollar,3
annual_price_growth_pct,3
price_growth_acceleration,70
high_growth_flag,3


## 2. Score Design and Missing-Value Treatment

The framework contains eight dimensions. Missing annual growth and acceleration
values arise from unavailable lags; they are assigned zero to represent no
observed change rather than being estimated from other counties or years.


In [3]:
SCORE_VARIABLES = {
    "housing_pressure": [
        "annual_price_growth_pct",
        "appreciation_from_baseline_pct",
        "annual_price_volatility",
        "price_growth_acceleration",
    ],
    "affordability_stress": [
        "price_to_income_ratio",
        "median_gross_rent",
        "median_household_income",
    ],
    "climate_exposure": ["CFLD_RISKS", "HRCN_RISKS"],
    "disaster_history": [
        "climate_disaster_count",
        "cumulative_climate_disaster_count",
        "recent_3yr_climate_disaster_count",
        "hurricane_disaster_count",
    ],
    "socioeconomic_vulnerability": [
        "SOVI_SCORE",
        "poverty_rate",
        "unemployment_rate",
        "renter_occupied_share",
    ],
    "resilience_adjustment": ["RESL_SCORE"],
    "insurance_loss_stress": [
        "nfip_claim_count",
        "nfip_total_claim_payment",
        "nfip_cumulative_claim_payment",
        "nfip_recent_3yr_claim_payment",
        "nfip_avg_claim_payment",
    ],
    "spatial_spillover": [
        "neighbor_avg_price_growth_pct",
        "neighbor_avg_price_volatility",
        "neighbor_high_growth_share",
        "neighbor_high_volatility_share",
    ],
}

all_score_variables = list(
    dict.fromkeys(
        variable
        for variables in SCORE_VARIABLES.values()
        for variable in variables
    )
)
missing_score_columns = sorted(
    set(all_score_variables) - set(county_year_features.columns)
)
if missing_score_columns:
    raise KeyError(f"Required score variables are missing: {missing_score_columns}")

score_design = pd.DataFrame(
    [
        {
            "Sub-score": group,
            "Component count": len(variables),
            "Variables": ", ".join(variables),
        }
        for group, variables in SCORE_VARIABLES.items()
    ]
)

vulnerability_df = county_year_features.copy()
score_fill_columns = [
    "annual_price_growth_pct",
    "price_growth_acceleration",
]
missing_before_fill = vulnerability_df[score_fill_columns].isna().sum()
vulnerability_df[score_fill_columns] = vulnerability_df[
    score_fill_columns
].fillna(0)

if vulnerability_df[all_score_variables].isna().any().any():
    raise ValueError("Missing values remain in variables used for scoring.")

score_summary = vulnerability_df[all_score_variables].describe().T
score_summary["skew"] = vulnerability_df[all_score_variables].skew()

display(score_design)
display(missing_before_fill.rename("filled_with_zero").to_frame())
display(score_summary.round(3))


,Sub-score,Component count,Variables
0,housing_pressure,4,"annual_price_growth_pct, appreciation_from_bas..."
1,affordability_stress,3,"price_to_income_ratio, median_gross_rent, medi..."
2,climate_exposure,2,"CFLD_RISKS, HRCN_RISKS"
3,disaster_history,4,"climate_disaster_count, cumulative_climate_dis..."
4,socioeconomic_vulnerability,4,"SOVI_SCORE, poverty_rate, unemployment_rate, r..."
5,resilience_adjustment,1,RESL_SCORE
6,insurance_loss_stress,5,"nfip_claim_count, nfip_total_claim_payment, nf..."
7,spatial_spillover,4,"neighbor_avg_price_growth_pct, neighbor_avg_pr..."


,filled_with_zero
annual_price_growth_pct,3
price_growth_acceleration,70


,count,mean,std,min,25%,50%,75%,max,skew
annual_price_growth_pct,"1,000.000",6.271,7.948,-14.023,1.316,5.903,9.865,35.736,0.510
appreciation_from_baseline_pct,"1,000.000",57.045,61.679,-19.571,3.414,39.777,102.412,267.024,0.837
annual_price_volatility,"1,000.000","5,510.032","5,890.973",161.593,"2,161.468","3,834.138","6,172.371","57,938.922",3.388
price_growth_acceleration,"1,000.000",0.310,7.387,-31.271,-2.266,0.055,3.969,19.183,-1.090
price_to_income_ratio,"1,000.000",3.903,1.261,1.855,2.970,3.728,4.544,11.815,1.670
median_gross_rent,"1,000.000",995.817,313.097,532.000,755.000,937.500,"1,143.000","2,098.700",1.072
median_household_income,"1,000.000","52,710.890","14,197.134","29,806.000","41,522.500","49,682.000","60,779.500","118,559.100",0.947
CFLD_RISKS,"1,000.000",47.536,37.729,0.000,0.000,61.200,78.550,99.600,-0.106
HRCN_RISKS,"1,000.000",94.015,5.436,74.468,90.572,94.994,98.665,99.958,-1.166
climate_disaster_count,"1,000.000",1.193,1.493,0.000,0.000,1.000,2.000,7.000,1.524


## 3. Transformations, Scaling, and Orientation

Nine right-skewed non-negative variables receive a `log(1+x)` transformation.
All scoring components are then min–max scaled to 0–1 using the complete
2011–2025 panel. Income and resilience are reversed after scaling so that every
component has the same direction: higher values indicate greater vulnerability.

Full-panel scaling is intentional because the score and classes describe the
completed historical panel; the later-period evaluation reproduces these
labels rather than representing a prospective forecast.


In [4]:
LOG_TRANSFORM_VARIABLES = [
    "annual_price_volatility",
    "price_to_income_ratio",
    "median_gross_rent",
    "neighbor_avg_price_volatility",
    "nfip_claim_count",
    "nfip_total_claim_payment",
    "nfip_cumulative_claim_payment",
    "nfip_recent_3yr_claim_payment",
    "nfip_avg_claim_payment",
]

score_df = vulnerability_df.copy()
for variable in LOG_TRANSFORM_VARIABLES:
    if score_df[variable].lt(0).any():
        raise ValueError(f"Log-transformed variable contains negatives: {variable}")
    score_df[f"log_{variable}"] = np.log1p(score_df[variable])

FINAL_SCORE_VARIABLES = {
    "housing_pressure": [
        "annual_price_growth_pct",
        "appreciation_from_baseline_pct",
        "log_annual_price_volatility",
        "price_growth_acceleration",
    ],
    "affordability_stress": [
        "log_price_to_income_ratio",
        "log_median_gross_rent",
        "median_household_income",
    ],
    "climate_exposure": ["CFLD_RISKS", "HRCN_RISKS"],
    "disaster_history": [
        "climate_disaster_count",
        "cumulative_climate_disaster_count",
        "recent_3yr_climate_disaster_count",
        "hurricane_disaster_count",
    ],
    "socioeconomic_vulnerability": [
        "SOVI_SCORE",
        "poverty_rate",
        "unemployment_rate",
        "renter_occupied_share",
    ],
    "resilience_adjustment": ["RESL_SCORE"],
    "insurance_loss_stress": [
        "log_nfip_claim_count",
        "log_nfip_total_claim_payment",
        "log_nfip_cumulative_claim_payment",
        "log_nfip_recent_3yr_claim_payment",
        "log_nfip_avg_claim_payment",
    ],
    "spatial_spillover": [
        "neighbor_avg_price_growth_pct",
        "log_neighbor_avg_price_volatility",
        "neighbor_high_growth_share",
        "neighbor_high_volatility_share",
    ],
}

final_score_variable_list = list(
    dict.fromkeys(
        variable
        for variables in FINAL_SCORE_VARIABLES.values()
        for variable in variables
    )
)
scaled_score_columns = [
    f"scaled_{variable}" for variable in final_score_variable_list
]

scaler = MinMaxScaler()
scaled_df = score_df.copy()
scaled_df[scaled_score_columns] = scaler.fit_transform(
    scaled_df[final_score_variable_list]
)

scored_df = scaled_df.copy()
scored_df["scaled_low_income_stress"] = (
    1 - scored_df["scaled_median_household_income"]
)
scored_df["scaled_low_resilience"] = 1 - scored_df["scaled_RESL_SCORE"]

scaling_audit = pd.DataFrame(
    {
        "Variable": final_score_variable_list,
        "Scaled column": scaled_score_columns,
        "Minimum before scaling": scaler.data_min_,
        "Maximum before scaling": scaler.data_max_,
        "Log transformed": [
            variable.startswith("log_")
            for variable in final_score_variable_list
        ],
        "Protective orientation": [
            variable in {"median_household_income", "RESL_SCORE"}
            for variable in final_score_variable_list
        ],
    }
)

SCALING_TOLERANCE = 1e-12
if scored_df[scaled_score_columns].min().min() < -SCALING_TOLERANCE:
    raise ValueError("A scaled component is below zero.")
if scored_df[scaled_score_columns].max().max() > 1 + SCALING_TOLERANCE:
    raise ValueError("A scaled component exceeds one.")

display(scaling_audit)
display(
    scored_df[
        [
            "scaled_median_household_income",
            "scaled_low_income_stress",
            "scaled_RESL_SCORE",
            "scaled_low_resilience",
        ]
    ].describe().T.round(3)
)


,Variable,Scaled column,Minimum before scaling,Maximum before scaling,Log transformed,Protective orientation
0,annual_price_growth_pct,scaled_annual_price_growth_pct,-14.023,35.736,False,False
1,appreciation_from_baseline_pct,scaled_appreciation_from_baseline_pct,-19.571,267.024,False,False
2,log_annual_price_volatility,scaled_log_annual_price_volatility,5.091,10.967,True,False
3,price_growth_acceleration,scaled_price_growth_acceleration,-31.271,19.183,False,False
4,log_price_to_income_ratio,scaled_log_price_to_income_ratio,1.049,2.551,True,False
5,log_median_gross_rent,scaled_log_median_gross_rent,6.279,7.650,True,False
6,median_household_income,scaled_median_household_income,"29,806.000","118,559.100",False,True
7,CFLD_RISKS,scaled_CFLD_RISKS,0.000,99.600,False,False
8,HRCN_RISKS,scaled_HRCN_RISKS,74.468,99.958,False,False
9,climate_disaster_count,scaled_climate_disaster_count,0.000,7.000,False,False


,count,mean,std,min,25%,50%,75%,max
scaled_median_household_income,"1,000.000",0.258,0.160,0.000,0.132,0.224,0.349,1.000
scaled_low_income_stress,"1,000.000",0.742,0.160,0.000,0.651,0.776,0.868,1.000
scaled_RESL_SCORE,"1,000.000",0.331,0.239,0.000,0.107,0.301,0.493,1.000
scaled_low_resilience,"1,000.000",0.669,0.239,0.000,0.507,0.699,0.893,1.000


## 4. Vulnerability Sub-Scores

Each sub-score is the unweighted mean of its oriented 0–1 components. This
retains the eight conceptual dimensions without allowing groups containing more
variables to receive greater weight in the final score.


In [5]:
SUB_SCORE_COMPONENTS = {
    "housing_pressure_score": [
        "scaled_annual_price_growth_pct",
        "scaled_appreciation_from_baseline_pct",
        "scaled_log_annual_price_volatility",
        "scaled_price_growth_acceleration",
    ],
    "affordability_stress_score": [
        "scaled_log_price_to_income_ratio",
        "scaled_log_median_gross_rent",
        "scaled_low_income_stress",
    ],
    "climate_exposure_score": [
        "scaled_CFLD_RISKS",
        "scaled_HRCN_RISKS",
    ],
    "disaster_history_score": [
        "scaled_climate_disaster_count",
        "scaled_cumulative_climate_disaster_count",
        "scaled_recent_3yr_climate_disaster_count",
        "scaled_hurricane_disaster_count",
    ],
    "socioeconomic_vulnerability_score": [
        "scaled_SOVI_SCORE",
        "scaled_poverty_rate",
        "scaled_unemployment_rate",
        "scaled_renter_occupied_share",
    ],
    "resilience_adjustment_score": ["scaled_low_resilience"],
    "insurance_loss_stress_score": [
        "scaled_log_nfip_claim_count",
        "scaled_log_nfip_total_claim_payment",
        "scaled_log_nfip_cumulative_claim_payment",
        "scaled_log_nfip_recent_3yr_claim_payment",
        "scaled_log_nfip_avg_claim_payment",
    ],
    "spatial_spillover_score": [
        "scaled_neighbor_avg_price_growth_pct",
        "scaled_log_neighbor_avg_price_volatility",
        "scaled_neighbor_high_growth_share",
        "scaled_neighbor_high_volatility_share",
    ],
}

for score_name, components in SUB_SCORE_COMPONENTS.items():
    scored_df[score_name] = scored_df[components].mean(axis=1)

sub_score_columns = list(SUB_SCORE_COMPONENTS)
sub_score_summary = scored_df[sub_score_columns].describe().T

if scored_df[sub_score_columns].isna().any().any():
    raise ValueError("Missing sub-score values found.")
if scored_df[sub_score_columns].min().min() < -SCALING_TOLERANCE:
    raise ValueError("A vulnerability sub-score is below zero.")
if scored_df[sub_score_columns].max().max() > 1 + SCALING_TOLERANCE:
    raise ValueError("A vulnerability sub-score exceeds one.")

display(sub_score_summary.round(3))


,count,mean,std,min,25%,50%,75%,max
housing_pressure_score,"1,000.000",0.459,0.110,0.247,0.389,0.449,0.500,0.857
affordability_stress_score,"1,000.000",0.502,0.076,0.295,0.448,0.496,0.555,0.787
climate_exposure_score,"1,000.000",0.622,0.260,0.000,0.390,0.617,0.862,1.000
disaster_history_score,"1,000.000",0.246,0.227,0.000,0.031,0.219,0.372,1.000
socioeconomic_vulnerability_score,"1,000.000",0.418,0.127,0.056,0.335,0.419,0.494,0.771
resilience_adjustment_score,"1,000.000",0.669,0.239,0.000,0.507,0.699,0.893,1.000
insurance_loss_stress_score,"1,000.000",0.438,0.237,0.000,0.237,0.505,0.619,0.984
spatial_spillover_score,"1,000.000",0.477,0.252,0.059,0.257,0.456,0.684,0.987


## 5. Final Score and Vulnerability Classes

The overall score is the unweighted mean of the eight sub-scores. Tertile
cut-points from the complete panel define the Low, Medium, and High classes.


In [6]:
scored_df["climate_housing_vulnerability_score"] = scored_df[
    sub_score_columns
].mean(axis=1)

scored_df["vulnerability_class"] = pd.qcut(
    scored_df["climate_housing_vulnerability_score"],
    q=3,
    labels=["Low", "Medium", "High"],
)

class_summary = (
    scored_df.groupby("vulnerability_class", observed=True)[
        "climate_housing_vulnerability_score"
    ]
    .agg(["count", "min", "mean", "max"])
    .reset_index()
)

ranking_columns = [
    "RegionName",
    "STCOFIPS",
    "Year",
    "climate_housing_vulnerability_score",
    "vulnerability_class",
    *sub_score_columns,
]
top_vulnerable_county_years = (
    scored_df.nlargest(15, "climate_housing_vulnerability_score")[
        ranking_columns
    ]
)
low_vulnerable_county_years = (
    scored_df.nsmallest(15, "climate_housing_vulnerability_score")[
        ranking_columns
    ]
)

display(
    scored_df[["climate_housing_vulnerability_score"]]
    .describe()
    .T
    .round(3)
)
display(class_summary.round(3))
display(top_vulnerable_county_years.round(3))
display(low_vulnerable_county_years.round(3))


,count,mean,std,min,25%,50%,75%,max
climate_housing_vulnerability_score,"1,000.000",0.479,0.093,0.187,0.409,0.477,0.546,0.782


,vulnerability_class,count,min,mean,max
0,Low,334,0.187,0.376,0.438
1,Medium,333,0.439,0.478,0.519
2,High,333,0.520,0.582,0.782


,RegionName,STCOFIPS,Year,climate_housing_vulnerability_score,vulnerability_class,housing_pressure_score,affordability_stress_score,climate_exposure_score,disaster_history_score,socioeconomic_vulnerability_score,resilience_adjustment_score,insurance_loss_stress_score,spatial_spillover_score
521,Lee County,12071,2022,0.782,High,0.838,0.616,0.988,0.603,0.357,0.915,0.984,0.959
861,Sarasota County,12115,2022,0.749,High,0.815,0.616,0.962,0.633,0.290,0.911,0.800,0.967
641,Miami-Dade County,12086,2022,0.742,High,0.755,0.704,1.000,0.622,0.551,0.558,0.759,0.987
161,Collier County,12021,2022,0.740,High,0.851,0.636,0.974,0.656,0.352,0.575,0.923,0.952
116,Charlotte County,12015,2022,0.738,High,0.827,0.606,0.935,0.591,0.328,0.880,0.796,0.938
191,De Soto County,12027,2022,0.730,High,0.790,0.577,0.595,0.580,0.640,0.992,0.719,0.943
596,Manatee County,12081,2022,0.724,High,0.834,0.624,0.980,0.675,0.304,0.769,0.673,0.934
771,Pinellas County,12103,2022,0.724,High,0.731,0.618,0.977,0.645,0.361,0.893,0.627,0.941
651,Monroe County,12087,2022,0.719,High,0.768,0.787,0.958,0.406,0.374,0.700,0.785,0.975
741,Palm Beach County,12099,2022,0.712,High,0.787,0.639,0.967,0.625,0.370,0.751,0.641,0.914


,RegionName,STCOFIPS,Year,climate_housing_vulnerability_score,vulnerability_class,housing_pressure_score,affordability_stress_score,climate_exposure_score,disaster_history_score,socioeconomic_vulnerability_score,resilience_adjustment_score,insurance_loss_stress_score,spatial_spillover_score
15,Baker County,12003,2011,0.187,Low,0.307,0.396,0.000,0.000,0.338,0.256,0.000,0.202
19,Baker County,12003,2015,0.242,Low,0.408,0.411,0.000,0.011,0.316,0.256,0.125,0.410
18,Baker County,12003,2014,0.249,Low,0.391,0.415,0.000,0.031,0.339,0.256,0.249,0.313
17,Baker County,12003,2013,0.256,Low,0.398,0.380,0.000,0.031,0.329,0.256,0.249,0.405
16,Baker County,12003,2012,0.257,Low,0.330,0.393,0.000,0.066,0.329,0.256,0.605,0.081
925,Union County,12125,2011,0.259,Low,0.307,0.329,0.137,0.000,0.454,0.672,0.000,0.169
0,Alachua County,12001,2011,0.261,Low,0.316,0.513,0.436,0.000,0.529,0.160,0.000,0.136
27,Baker County,12003,2023,0.266,Low,0.335,0.465,0.000,0.418,0.179,0.256,0.144,0.333
955,Wakulla County,12129,2011,0.269,Low,0.290,0.407,0.504,0.000,0.204,0.565,0.000,0.182
480,Lafayette County,12067,2011,0.271,Low,0.299,0.295,0.305,0.000,0.310,0.884,0.000,0.074


## 6. Final Validation and Export

The final dataset retains all 75 input variables and adds nine transformed
variables, 29 scaled/oriented variables, eight sub-scores, the overall score,
and the vulnerability class.


In [7]:
expected_class_counts = {"Low": 334, "Medium": 333, "High": 333}
actual_class_counts = (
    scored_df["vulnerability_class"]
    .value_counts()
    .reindex(["Low", "Medium", "High"])
    .astype(int)
    .to_dict()
)

final_validation = pd.DataFrame(
    {
        "Metric": [
            "Rows",
            "Columns",
            "Counties",
            "Start year",
            "End year",
            "Duplicate county-years",
            "Missing overall scores",
            "Minimum overall score",
            "Maximum overall score",
        ],
        "Value": [
            len(scored_df),
            scored_df.shape[1],
            scored_df["STCOFIPS"].nunique(),
            scored_df["Year"].min(),
            scored_df["Year"].max(),
            int(scored_df.duplicated(["STCOFIPS", "Year"]).sum()),
            int(scored_df["climate_housing_vulnerability_score"].isna().sum()),
            scored_df["climate_housing_vulnerability_score"].min(),
            scored_df["climate_housing_vulnerability_score"].max(),
        ],
    }
)

if scored_df.shape != (1000, 123):
    raise ValueError("Expected a 1,000-row × 123-column vulnerability dataset.")
if actual_class_counts != expected_class_counts:
    raise ValueError(
        f"Unexpected class distribution: {actual_class_counts}"
    )
if scored_df.duplicated(["STCOFIPS", "Year"]).any():
    raise ValueError("Duplicate county-year observations found after scoring.")
if scored_df["climate_housing_vulnerability_score"].isna().any():
    raise ValueError("Missing final vulnerability scores found.")

OUTPUT_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
scored_df.to_csv(OUTPUT_DATA_PATH, index=False)

save_table(
    score_design,
    TABLES_DIR / "vulnerability_score_design.csv",
)
save_table(
    scaling_audit,
    TABLES_DIR / "vulnerability_scaling_audit.csv",
)
save_table(
    sub_score_summary.reset_index().rename(columns={"index": "Sub-score"}),
    TABLES_DIR / "vulnerability_subscore_summary.csv",
)
save_table(
    class_summary,
    TABLES_DIR / "vulnerability_class_summary.csv",
)
save_table(
    top_vulnerable_county_years,
    TABLES_DIR / "top_vulnerability_county_years.csv",
)
save_table(
    low_vulnerable_county_years,
    TABLES_DIR / "low_vulnerability_county_years.csv",
)

display(final_validation.round(3))
print(f"Saved: {OUTPUT_DATA_PATH.relative_to(PROJECT_ROOT)}")


Saved table: C:\Users\saadm\OneDrive\Documents\MRP\climate-risk-housing-valuation\results\tables\vulnerability_scoring\vulnerability_score_design.csv
Saved table: C:\Users\saadm\OneDrive\Documents\MRP\climate-risk-housing-valuation\results\tables\vulnerability_scoring\vulnerability_scaling_audit.csv
Saved table: C:\Users\saadm\OneDrive\Documents\MRP\climate-risk-housing-valuation\results\tables\vulnerability_scoring\vulnerability_subscore_summary.csv
Saved table: C:\Users\saadm\OneDrive\Documents\MRP\climate-risk-housing-valuation\results\tables\vulnerability_scoring\vulnerability_class_summary.csv
Saved table: C:\Users\saadm\OneDrive\Documents\MRP\climate-risk-housing-valuation\results\tables\vulnerability_scoring\top_vulnerability_county_years.csv
Saved table: C:\Users\saadm\OneDrive\Documents\MRP\climate-risk-housing-valuation\results\tables\vulnerability_scoring\low_vulnerability_county_years.csv


,Metric,Value
0,Rows,"1,000.000"
1,Columns,123.000
2,Counties,67.000
3,Start year,"2,011.000"
4,End year,"2,025.000"
5,Duplicate county-years,0.000
6,Missing overall scores,0.000
7,Minimum overall score,0.187
8,Maximum overall score,0.782


Saved: data\processed\florida_county_year_vulnerability_2011_2025.csv


The completed vulnerability dataset contains 1,000 county-year observations
and 123 variables. The class distribution is deliberately near-balanced: 334
Low, 333 Medium, and 333 High observations. Notebook 05 uses these labels for
the later-period supervised classification analysis.
